In [1]:

import pandas as pd
import numpy as np
import random
from copy import deepcopy

# 设置随机种子
np.random.seed(42)
random.seed(42)

# 重新加载数据并检查
print("=== 检查供应商数据 ===")
df_prediction = pd.read_excel('期望供货量预测结果.xlsx')
print("供应商数据形状:", df_prediction.shape)
print("材料分类的唯一值:", df_prediction['材料分类'].unique())
print("材料分类的缺失值数量:", df_prediction['材料分类'].isna().sum())
print("供应商ID的缺失值数量:", df_prediction['供应商ID'].isna().sum())

# 预处理数据
suppliers = df_prediction[['供应商ID', '材料分类']].copy()
supplier_ids = suppliers['供应商ID'].tolist()

# 创建供应商材料类型字典，处理可能的缺失值
supplier_materials = {}
for _, row in suppliers.iterrows():
    supplier_id = row['供应商ID']
    material_type = row['材料分类']
    if pd.isna(material_type):
        # 如果材料分类缺失，默认为B类
        supplier_materials[supplier_id] = 'B'
    else:
        supplier_materials[supplier_id] = material_type

week_columns = [f'未来第{i}周' for i in range(1, 25)]

# 创建供应商期望供货量的字典
expected_supply_dict = {}
for _, row in df_prediction.iterrows():
    supplier_id = row['供应商ID']
    expected_supply_dict[supplier_id] = {}
    for week in range(1, 25):
        week_col = f'未来第{week}周'
        expected_supply_dict[supplier_id][week] = row[week_col]

# 转运商数据
df_transport = pd.read_excel('附件2.xlsx')
transport_weeks = [col for col in df_transport.columns if col.startswith('W')]
transport_avg_loss = {}
for idx, row in df_transport.iterrows():
    transport_id = row['转运商ID']
    loss_rates = row[transport_weeks].values
    avg_loss = np.mean(loss_rates)
    transport_avg_loss[transport_id] = avg_loss

transport_ids = list(transport_avg_loss.keys())
transport_capacity = 6000  # 每家转运商每周运输能力

# 问题参数
WEEKS = 24
CAPACITY = 28200
SAFETY_STOCK_WEEKS = 2

CONSUMPTION_RATIOS = {'A': 0.6, 'B': 0.66, 'C': 0.72}
EQUIVALENT_COEFFICIENTS = {
    'A': 1.0,
    'B': CONSUMPTION_RATIOS['B'] / CONSUMPTION_RATIOS['A'],
    'C': CONSUMPTION_RATIOS['C'] / CONSUMPTION_RATIOS['A']
}

base_material_demand = CAPACITY * CONSUMPTION_RATIOS['A']
safety_stock_level = SAFETY_STOCK_WEEKS * base_material_demand

print(f"\n材料分类分布:")
for material in ['A', 'B', 'C']:
    count = sum(1 for mat in supplier_materials.values() if mat == material)
    print(f"{material}类: {count}家供应商")

print(f"\n转运商平均损耗率:")
for transport_id in sorted(transport_ids):
    print(f"{transport_id}: {transport_avg_loss[transport_id]:.4f}%")

print(f"\n每周等效材料需求: {base_material_demand:.2f} 立方米")
print(f"安全库存水平: {safety_stock_level:.2f} 立方米")

# 简化遗传算法实现
print("\n=== 实现更简化的遗传算法 ===")

def create_individual():
    """创建一个个体，代表24周的订购和转运方案"""
    individual = {
        'orders': {},  # {供应商ID: {周数: 订货量}}
        'transporters': {}  # {供应商ID: {周数: 转运商ID}}
    }

    for supplier_id in supplier_ids:
        individual['orders'][supplier_id] = {}
        individual['transporters'][supplier_id] = {}

        for week in range(1, WEEKS + 1):
            # 获取期望供货量
            exp_supply = expected_supply_dict[supplier_id][week]
            material_type = supplier_materials[supplier_id]

            # 基于材料类型和期望供货量决定是否订购
            if material_type == 'A':
                # A类材料优先订购
                if exp_supply > 0 and random.random() < 0.8:
                    individual['orders'][supplier_id][week] = exp_supply
                    # 优先分配损耗率最低的转运商
                    best_transport = min(transport_ids, key=lambda x: transport_avg_loss[x])
                    individual['transporters'][supplier_id][week] = best_transport
                else:
                    individual['orders'][supplier_id][week] = 0
                    individual['transporters'][supplier_id][week] = None
            elif material_type == 'B':
                # B类材料正常订购
                if exp_supply > 0 and random.random() < 0.5:
                    individual['orders'][supplier_id][week] = exp_supply
                    # 分配损耗率较低的转运商
                    sorted_transports = sorted(transport_ids, key=lambda x: transport_avg_loss[x])
                    transport_id = random.choices(sorted_transports[:4], weights=[0.5, 0.3, 0.15, 0.05], k=1)[0]
                    individual['transporters'][supplier_id][week] = transport_id
                else:
                    individual['orders'][supplier_id][week] = 0
                    individual['transporters'][supplier_id][week] = None
            else:  # C类
                # C类材料尽量少订购
                if exp_supply > 0 and random.random() < 0.2:
                    individual['orders'][supplier_id][week] = exp_supply
                    # 可以分配任何转运商
                    transport_id = random.choice(transport_ids)
                    individual['transporters'][supplier_id][week] = transport_id
                else:
                    individual['orders'][supplier_id][week] = 0
                    individual['transporters'][supplier_id][week] = None

    return individual

def calculate_fitness(individual):
    """计算个体的适应度"""
    # 初始化变量
    total_a_amount = 0
    total_b_amount = 0
    total_c_amount = 0
    total_ordered = 0
    total_received = 0

    # 库存跟踪
    inventory = safety_stock_level
    inventory_penalty = 0

    # 转运商使用量
    transport_usage = {t_id: 0 for t_id in transport_ids}

    # 每周计算
    for week in range(1, WEEKS + 1):
        week_ordered = 0
        week_received = 0
        week_a = 0
        week_b = 0
        week_c = 0
        week_equivalent = 0

        # 重置转运商使用量
        weekly_transport_usage = {t_id: 0 for t_id in transport_ids}

        # 处理每个供应商
        for supplier_id in supplier_ids:
            order_qty = individual['orders'][supplier_id][week]
            if order_qty > 0:
                transport_id = individual['transporters'][supplier_id][week]
                material_type = supplier_materials[supplier_id]

                # 应用损耗率
                loss_rate = transport_avg_loss[transport_id]
                received_qty = order_qty * (1 - loss_rate / 100)

                # 计算等效材料量
                equivalent_qty = received_qty * EQUIVALENT_COEFFICIENTS[material_type]

                # 更新统计
                week_ordered += order_qty
                week_received += received_qty
                week_equivalent += equivalent_qty

                if material_type == 'A':
                    week_a += received_qty
                elif material_type == 'B':
                    week_b += received_qty
                else:
                    week_c += received_qty

                # 更新转运商使用量
                weekly_transport_usage[transport_id] += received_qty

        # 更新全局统计
        total_ordered += week_ordered
        total_received += week_received
        total_a_amount += week_a
        total_b_amount += week_b
        total_c_amount += week_c

        # 库存管理
        inventory += week_equivalent
        inventory -= base_material_demand

        # 检查库存约束
        if inventory < safety_stock_level:
            inventory_penalty += (safety_stock_level - inventory) * 10

        # 确保库存不为负
        inventory = max(0, inventory)

        # 检查转运商容量约束
        for t_id, usage in weekly_transport_usage.items():
            if usage > transport_capacity:
                inventory_penalty += (usage - transport_capacity) * 5

    # 计算各类材料比例
    if total_received > 0:
        a_ratio = total_a_amount / total_received
        b_ratio = total_b_amount / total_received
        c_ratio = total_c_amount / total_received
    else:
        a_ratio = 0
        b_ratio = 0
        c_ratio = 0

    # 计算总损耗率
    if total_ordered > 0:
        total_loss_rate = 1 - total_received / total_ordered
    else:
        total_loss_rate = 0

    # 计算适应度 - 权重可以调整
    fitness = (a_ratio * 100) - (c_ratio * 50) - (total_loss_rate * 1000) - (inventory_penalty / 10000)

    return {
        'fitness': fitness,
        'a_ratio': a_ratio,
        'b_ratio': b_ratio,
        'c_ratio': c_ratio,
        'loss_rate': total_loss_rate,
        'total_ordered': total_ordered,
        'total_received': total_received
    }

def select_parents(population, fitness_scores):
    """选择父母进行交叉"""
    fitness_values = [f['fitness'] for f in fitness_scores]
    total_fitness = sum(fitness_values)

    if total_fitness <= 0 or len(population) < 2:
        return random.sample(population, min(2, len(population)))

    probabilities = [f / total_fitness for f in fitness_values]
    parent1 = random.choices(population, weights=probabilities, k=1)[0]
    parent2 = random.choices(population, weights=probabilities, k=1)[0]

    return parent1, parent2

def crossover(parent1, parent2):
    """简单交叉"""
    child = deepcopy(parent1)

    # 随机选择一个交叉点（周）
    crossover_week = random.randint(1, WEEKS)

    # 交叉点之后的周数从父2继承
    for week in range(crossover_week, WEEKS + 1):
        for supplier_id in supplier_ids:
            if random.random() < 0.6:  # 60%概率从父2继承
                child['orders'][supplier_id][week] = parent2['orders'][supplier_id][week]
                child['transporters'][supplier_id][week] = parent2['transporters'][supplier_id][week]

    return child

def mutate(individual):
    """变异操作"""
    for week in range(1, WEEKS + 1):
        for supplier_id in supplier_ids:
            if random.random() < MUTATION_RATE:
                # 变异订货量
                current_order = individual['orders'][supplier_id][week]
                material_type = supplier_materials[supplier_id]
                exp_supply = expected_supply_dict[supplier_id][week]

                if current_order > 0:
                    # 如果已有订单，有概率取消或调整
                    if random.random() < 0.3:
                        individual['orders'][supplier_id][week] = 0
                        individual['transporters'][supplier_id][week] = None
                    else:
                        # 调整订货量
                        new_qty = max(1, current_order * random.uniform(0.8, 1.2))
                        individual['orders'][supplier_id][week] = new_qty
                else:
                    # 如果没有订单，有概率新增
                    prob_threshold = 0.7 if material_type == 'A' else (0.4 if material_type == 'B' else 0.1)
                    if exp_supply > 0 and random.random() < prob_threshold:
                        individual['orders'][supplier_id][week] = exp_supply
                        # 分配转运商
                        if material_type == 'A':
                            best_transport = min(transport_ids, key=lambda x: transport_avg_loss[x])
                            individual['transporters'][supplier_id][week] = best_transport
                        else:
                            transport_id = random.choice(transport_ids)
                            individual['transporters'][supplier_id][week] = transport_id

            elif random.random() < MUTATION_RATE and individual['orders'][supplier_id][week] > 0:
                # 变异转运商分配
                transport_id = random.choice(transport_ids)
                individual['transporters'][supplier_id][week] = transport_id

    return individual

def genetic_algorithm():
    """遗传算法主函数"""
    print("开始遗传算法优化...")

    # 初始化种群
    population = [create_individual() for _ in range(POPULATION_SIZE)]
    best_fitness_history = []
    avg_fitness_history = []

    for generation in range(GENERATIONS):
        # 计算适应度
        fitness_scores = [calculate_fitness(individual) for individual in population]

        # 记录最佳和平均适应度
        fitness_values = [f['fitness'] for f in fitness_scores]
        best_idx = np.argmax(fitness_values)
        best_fitness = fitness_values[best_idx]
        avg_fitness = np.mean(fitness_values)

        best_fitness_history.append(best_fitness)
        avg_fitness_history.append(avg_fitness)

        # 输出当前代的信息
        if generation % 10 == 0 or generation == GENERATIONS - 1:
            best_stats = fitness_scores[best_idx]
            print(f"第 {generation} 代: 最佳适应度 = {best_fitness:.4f}, A类比例 = {best_stats['a_ratio']:.4f}, C类比例 = {best_stats['c_ratio']:.4f}, 损耗率 = {best_stats['loss_rate']:.6f}")

        # 创建下一代
        new_population = []

        # 保留精英
        elite_count = max(1, int(ELITISM_RATE * POPULATION_SIZE))
        elite_indices = np.argsort(fitness_values)[::-1][:elite_count]
        for idx in elite_indices:
            new_population.append(deepcopy(population[idx]))

        # 生成剩余个体
        while len(new_population) < POPULATION_SIZE:
            if len(population) >= 2:
                parent1, parent2 = select_parents(population, fitness_scores)
                child = crossover(parent1, parent2)
                child = mutate(child)
                new_population.append(child)
            else:
                # 如果种群太小，直接复制现有个体
                new_population.append(deepcopy(population[0]))

    # 返回最佳个体
    final_fitness_scores = [calculate_fitness(individual) for individual in population]
    final_fitness_values = [f['fitness'] for f in final_fitness_scores]
    best_idx = np.argmax(final_fitness_values)
    best_individual = population[best_idx]
    best_stats = final_fitness_scores[best_idx]

    print("\n优化完成!")
    print(f"最佳适应度: {best_stats['fitness']:.4f}")
    print(f"A类材料比例: {best_stats['a_ratio']:.4f}")
    print(f"B类材料比例: {best_stats['b_ratio']:.4f}")
    print(f"C类材料比例: {best_stats['c_ratio']:.4f}")
    print(f"总损耗率: {best_stats['loss_rate']:.6f}")
    print(f"总订货量: {best_stats['total_ordered']:.2f} 立方米")
    print(f"总接收量: {best_stats['total_received']:.2f} 立方米")

    return best_individual, best_stats, best_fitness_history, avg_fitness_history

# 简化参数以提高效率
POPULATION_SIZE = 20
GENERATIONS = 30
MUTATION_RATE = 0.15
ELITISM_RATE = 0.2

# 运行遗传算法
best_individual, best_stats, best_fitness_history, avg_fitness_history = genetic_algorithm()



=== 检查供应商数据 ===
供应商数据形状: (403, 26)
材料分类的唯一值: ['B' 'A' 'C' nan]
材料分类的缺失值数量: 1
供应商ID的缺失值数量: 1

材料分类分布:
A类: 146家供应商
B类: 135家供应商
C类: 122家供应商

转运商平均损耗率:
T1: 1.9048%
T2: 0.9214%
T3: 0.0907%
T4: 0.6675%
T5: 0.9994%
T6: 0.4894%
T7: 2.0788%
T8: 0.8545%

每周等效材料需求: 16920.00 立方米
安全库存水平: 33840.00 立方米

=== 实现更简化的遗传算法 ===
开始遗传算法优化...
第 0 代: 最佳适应度 = -163.8557, A类比例 = 0.4888, C类比例 = 0.1189, 损耗率 = 0.003328
第 10 代: 最佳适应度 = -163.8557, A类比例 = 0.4888, C类比例 = 0.1189, 损耗率 = 0.003328
第 20 代: 最佳适应度 = -163.8557, A类比例 = 0.4888, C类比例 = 0.1189, 损耗率 = 0.003328
第 29 代: 最佳适应度 = -163.8557, A类比例 = 0.4888, C类比例 = 0.1189, 损耗率 = 0.003328

优化完成!
最佳适应度: -163.8557
A类材料比例: 0.4888
B类材料比例: 0.3923
C类材料比例: 0.1189
总损耗率: 0.003328
总订货量: 359270.00 立方米
总接收量: 358074.27 立方米
